# TSA Chapter 1: Case Study — Romania Quarterly GDP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/TSA/blob/main/TSA_ch1/TSA_ch1_case_gdp/TSA_ch1_case_gdp.ipynb)

This notebook demonstrates the full stationarity analysis pipeline on **Romania quarterly GDP** (Eurostat, 1995–2024):

1. **Raw GDP visualization** — identifying trend and structural shocks
2. **ADF and KPSS tests** — testing for unit roots on the GDP level
3. **Zivot-Andrews test** — unit root test with endogenous structural breakpoint
4. **Log-differencing** — transforming to stationarity, verifying with ADF/KPSS

**Data source:** Eurostat `namq_10_gdp` (chain-linked volumes, index 2015=100, seasonally and calendar adjusted)

In [ ]:
!pip install matplotlib numpy pandas statsmodels arch -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Style configuration
COLORS = {
    'blue': '#1A3A6E',
    'red': '#DC3545',
    'green': '#2E7D32',
    'gray': '#666666',
}

plt.rcParams.update({
    'axes.facecolor': 'none',
    'figure.facecolor': 'none',
    'savefig.transparent': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'lines.linewidth': 1.2,
    'axes.edgecolor': '#333333',
    'axes.linewidth': 0.8,
})

def save_chart(fig, name):
    fig.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True, dpi=150)
    fig.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    try:
        charts_path = os.path.join('..', '..', '..', 'charts', name)
        fig.savefig(f'{charts_path}.pdf', bbox_inches='tight', transparent=True, dpi=150)
        fig.savefig(f'{charts_path}.png', bbox_inches='tight', transparent=True, dpi=150)
    except Exception:
        pass
    print(f'Saved: {name}.pdf + .png')

def add_legend_below(ax, ncol=3):
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=ncol, frameon=False)

In [ ]:
# Download Romania quarterly GDP from Eurostat
# namq_10_gdp: chain-linked volumes, index 2015=100, seasonally & calendar adjusted

EUROSTAT_URL = (
    'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/'
    'namq_10_gdp/Q.CLV_I15.SCA.B1GQ.RO?format=TSV'
)

try:
    raw = pd.read_csv(EUROSTAT_URL, sep='\t')
    row = raw.iloc[0]
    data = {}
    for col in raw.columns[1:]:
        col_clean = col.strip()
        val_str = str(row[col]).strip().replace(' p', '').replace(' e', '').replace(' b', '')
        if val_str not in (':', ''):
            try:
                date = pd.Period(col_clean, freq='Q').start_time
                data[date] = float(val_str)
            except (ValueError, TypeError):
                pass
    gdp_series = pd.Series(data).sort_index()
    gdp_series = gdp_series[(gdp_series.index >= '1995-01-01') & (gdp_series.index <= '2024-12-31')]
    print(f'Eurostat GDP loaded: {len(gdp_series)} quarters, {gdp_series.index[0].date()} to {gdp_series.index[-1].date()}')
except Exception as e:
    print(f'Eurostat download failed ({e}), using FRED fallback')
    FRED_URL = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id=CLVMNACSCAB1GQRO'
    gdp_df = pd.read_csv(FRED_URL, parse_dates=['observation_date'])
    gdp_df = gdp_df.rename(columns={'observation_date': 'date', 'CLVMNACSCAB1GQRO': 'gdp'})
    gdp_df['gdp'] = pd.to_numeric(gdp_df['gdp'], errors='coerce')
    gdp_df = gdp_df.dropna().set_index('date')
    gdp_series = gdp_df['gdp']
    gdp_series = gdp_series[(gdp_series.index >= '1995-01-01') & (gdp_series.index <= '2024-12-31')]
    print(f'FRED GDP loaded: {len(gdp_series)} quarters')

quarters = gdp_series.index
values = gdp_series.values
print(f'\n{len(values)} quarterly observations, mean = {np.mean(values):.1f}')

In [ ]:
# Chart 1: ch1_case_gdp_raw — Raw GDP level with crisis annotations
mean_val = np.mean(values)

fig, ax = plt.subplots(figsize=(10, 4.5))

ax.plot(quarters, values, color=COLORS['blue'], linewidth=1.8, marker='o', markersize=2.5)
ax.fill_between(quarters, values, alpha=0.08, color=COLORS['blue'])

# Mean line
ax.axhline(y=mean_val, color=COLORS['red'], linewidth=1.2, linestyle='--',
           label=f'Mean = {mean_val:.1f}')

ax.set_title(r'Romania GDP Level — Quarterly 1995–2024, Non-stationary $I(1)$',
             fontsize=12, fontweight='bold', color=COLORS['red'])
ax.set_ylabel('GDP volume index (2015=100)')
ax.set_xlabel('Year')
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.set_ylim(0, values.max() * 1.12)

# Annotate crises
crises = [
    (pd.Timestamp('1999-01-01'), 'Crisis\n1999'),
    (pd.Timestamp('2009-01-01'), 'GFC\n2009'),
    (pd.Timestamp('2020-04-01'), 'COVID\n2020'),
]
for date, label in crises:
    idx = np.argmin(np.abs(quarters - date))
    val = values[idx]
    ax.annotate(label, xy=(quarters[idx], val),
                xytext=(0, 20), textcoords='offset points',
                fontsize=8, color=COLORS['red'], fontweight='bold', ha='center',
                arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.0))

ax.legend(loc='upper left', frameon=False, fontsize=9)

fig.text(0.5, -0.03,
         'Source: Eurostat namq_10_gdp (chain-linked volumes, 2015=100, SCA)',
         ha='center', fontsize=8, color=COLORS['gray'], style='italic')

fig.tight_layout()
save_chart(fig, 'ch1_case_gdp_raw')
plt.show()

In [ ]:
# Step 2: ADF and KPSS tests on GDP level
from arch.unitroot import ADF, KPSS

print('=' * 60)
print('STATIONARITY TESTS ON GDP LEVEL')
print('=' * 60)

# ADF test
adf = ADF(values, lags=4, trend='c')
print(f'\nADF Test (H0: unit root)')
print(f'  Statistic: {adf.stat:.2f}')
print(f'  P-value:   {adf.pvalue:.4f}')
print(f'  Lags:      {adf.lags}')
print(f'  Decision:  {"Reject H0" if adf.pvalue < 0.05 else "Fail to reject H0"} → {"stationary" if adf.pvalue < 0.05 else "unit root (non-stationary)"}')

# KPSS test
kpss = KPSS(values, trend='c', lags=-1)
print(f'\nKPSS Test (H0: stationary)')
print(f'  Statistic: {kpss.stat:.2f}')
print(f'  P-value:   {kpss.pvalue:.4f}')
print(f'  Decision:  {"Reject H0" if kpss.pvalue < 0.05 else "Fail to reject H0"} → {"non-stationary" if kpss.pvalue < 0.05 else "stationary"}')

print(f'\n→ Both tests agree: GDP level is NON-STATIONARY (I(1))')

In [ ]:
# Step 3: Zivot-Andrews test — unit root with structural breakpoint
from arch.unitroot import ZivotAndrews

za = ZivotAndrews(values, method='c', lags=8)
za_stat = za.stat
za_pval = za.pvalue

# Find breakpoint index
stats = za._all_stats
nobs = len(values)
trim = int(nobs * 0.15)
valid = stats[trim:nobs - trim]
bp_idx = trim + np.argmin(valid)
bp_date = quarters[bp_idx]
bp_label = f'{bp_date.year} Q{(bp_date.month - 1) // 3 + 1}'

print('=' * 60)
print('ZIVOT-ANDREWS TEST (Model C: shift in mean and trend)')
print('=' * 60)
print(f'  Statistic:  {za_stat:.3f}')
print(f'  P-value:    {za_pval:.3f}')
print(f'  Breakpoint: {bp_label}')
print(f'  Critical values: {za.critical_values}')
print(f'  Decision:   {"Reject H0" if za_pval < 0.05 else "Fail to reject H0"} → {"trend-break stationary" if za_pval < 0.05 else "unit root persists even with structural break"}')

# Chart: ch1_zivot_andrews
fig, ax = plt.subplots(figsize=(8, 2.8))

ax.plot(quarters, values, color=COLORS['blue'], linewidth=1.0,
        label='Romania GDP (volume index, 2015=100)')
ax.axvline(x=bp_date, color=COLORS['red'], linewidth=1.8, linestyle='--',
           label=f'Detected breakpoint: {bp_label}')

ax.axvspan(quarters[0], bp_date, alpha=0.06, color=COLORS['blue'])
ax.axvspan(bp_date, quarters[-1], alpha=0.06, color=COLORS['green'])

za_text = f'ZA stat = {za_stat:.2f}  (p = {za_pval:.2f})\nFail to reject $H_0$ (unit root)'
ax.text(0.02, 0.95, za_text, transform=ax.transAxes, fontsize=7,
        verticalalignment='top', color=COLORS['gray'],
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8,
                  edgecolor=COLORS['gray'], linewidth=0.5))

ax.set_title('Zivot-Andrews Test: Romania Quarterly GDP with Structural Breakpoint',
             fontsize=9, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('GDP (2015=100)')
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

add_legend_below(ax, ncol=2)

fig.text(0.5, -0.08, 'Source: Eurostat namq_10_gdp (chain-linked volumes, 2015=100, SCA)',
         ha='center', fontsize=6.5, color=COLORS['gray'], style='italic')

fig.tight_layout(rect=[0, 0.08, 1, 1])
save_chart(fig, 'ch1_zivot_andrews')
plt.show()

In [ ]:
# Step 4: Log-differencing — transform to stationarity
log_gdp = np.log(values)
dlog_gdp = np.diff(log_gdp) * 100   # percentage quarterly growth
dlog_quarters = quarters[1:]
mean_growth = np.mean(dlog_gdp)

# Tests on log(GDP)
adf_log = ADF(log_gdp, lags=4, trend='c')
kpss_log = KPSS(log_gdp, trend='c', lags=-1)

# Tests on Δlog(GDP)
adf_diff = ADF(dlog_gdp, lags=4, trend='c')
kpss_diff = KPSS(dlog_gdp, trend='c', lags=-1)

print('=' * 60)
print('TESTS ON log(GDP) AND Δlog(GDP)')
print('=' * 60)
print(f'\nlog(GDP):  ADF = {adf_log.stat:.2f} (p = {adf_log.pvalue:.3f}),  KPSS = {kpss_log.stat:.2f} (p = {kpss_log.pvalue:.3f})  → I(1)')
print(f'Δlog(GDP): ADF = {adf_diff.stat:.2f} (p = {adf_diff.pvalue:.4f}), KPSS = {kpss_diff.stat:.2f} (p = {kpss_diff.pvalue:.3f})  → I(0) ✓')

def fmt_p(p):
    if p < 0.001: return 'p < 0.001'
    elif p > 0.10: return 'p > 0.10'
    else: return f'p = {p:.2f}'

def fmt_stars(p):
    if p < 0.001: return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    return ''

# Chart: ch1_gdp_differencing — two panels
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6.5), height_ratios=[1, 1])

# Panel 1: log(GDP)
ax1.plot(quarters, log_gdp, color=COLORS['blue'], linewidth=1.2, marker='o', markersize=2)
ax1.fill_between(quarters, log_gdp, alpha=0.08, color=COLORS['blue'])
ax1.set_title(r'log(GDP) Romania — linear trend, non-stationary $I(1)$',
              fontsize=11, fontweight='bold', color=COLORS['red'])
ax1.set_ylabel('log GDP')
ax1.set_xlabel('Year')
ax1.set_ylim(log_gdp.min() - 0.05, log_gdp.max() + 0.1)
ax1.xaxis.set_major_locator(mdates.YearLocator(5))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

adf_s1 = f'ADF = {adf_log.stat:.2f}  ({fmt_p(adf_log.pvalue)}) — fail to reject $H_0$ (unit root)'
kpss_s1 = f'KPSS = {kpss_log.stat:.2f}{fmt_stars(kpss_log.pvalue)}  ({fmt_p(kpss_log.pvalue)}) — reject $H_0$ (non-stationary)'
ax1.text(0.5, -0.18, f'{adf_s1}  |  {kpss_s1}',
         transform=ax1.transAxes, ha='center', fontsize=7.5, color=COLORS['gray'])

# Panel 2: Δlog(GDP)
colors_bar = [COLORS['green'] if v >= 0 else COLORS['red'] for v in dlog_gdp]
ax2.bar(dlog_quarters, dlog_gdp, width=70, color=colors_bar, alpha=0.75)
ax2.axhline(y=mean_growth, color=COLORS['green'], linewidth=1.2, linestyle='--',
            label=f'Mean = {mean_growth:.1f}%/qtr')
ax2.set_title(r'$\Delta\log$(GDP) — quarterly economic growth, stationary $I(0)$',
              fontsize=11, fontweight='bold', color=COLORS['green'])
ax2.set_ylabel('Growth rate (%)')
ax2.set_xlabel('Year')
ax2.xaxis.set_major_locator(mdates.YearLocator(5))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax2.legend(loc='upper left', frameon=False, fontsize=8)

# Annotate crises on growth chart
def nearest_val(target_date):
    idx = np.argmin(np.abs(dlog_quarters - target_date))
    return dlog_gdp[idx]

ax2.annotate('2009', xy=(pd.Timestamp('2009-01-01'), nearest_val(pd.Timestamp('2009-01-01'))),
             xytext=(pd.Timestamp('2011-01-01'), -8),
             fontsize=8, color=COLORS['red'], fontweight='bold', ha='center',
             arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.0))
ax2.annotate('COVID\n2020', xy=(pd.Timestamp('2020-04-01'), nearest_val(pd.Timestamp('2020-04-01'))),
             xytext=(pd.Timestamp('2022-06-01'), -8),
             fontsize=8, color=COLORS['red'], fontweight='bold', ha='center',
             arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.0))

adf_s2 = f'ADF = {adf_diff.stat:.2f}{fmt_stars(adf_diff.pvalue)}  ({fmt_p(adf_diff.pvalue)}) — reject $H_0$ (stationary)'
kpss_s2 = f'KPSS = {kpss_diff.stat:.2f}  ({fmt_p(kpss_diff.pvalue)}) — fail to reject $H_0$ (stationary)'
ax2.text(0.5, -0.18, f'{adf_s2}  |  {kpss_s2}',
         transform=ax2.transAxes, ha='center', fontsize=7.5, color=COLORS['gray'])

fig.text(0.5, -0.02, 'Source: Eurostat namq_10_gdp (chain-linked volumes, 2015=100, SCA)',
         ha='center', fontsize=7, color=COLORS['gray'], style='italic')

fig.tight_layout(h_pad=2.5)
save_chart(fig, 'ch1_gdp_differencing')
plt.show()

print(f'\nConclusion: GDP_t ~ I(1), ln(GDP_t) ~ I(1), Δln(GDP_t) ~ I(0) ✓')
print(f'Mean quarterly growth rate: {mean_growth:.2f}%')